Importation des bibliothèques

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import random_split
from tqdm import tqdm # pour la progression 
import time #pour le temps de calcul
import copy #pour copier base_model

In [2]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [3]:
batch_size = 128 #on peut changer la taille mais 128 est bien pour éviter underfitting ou overfitting

transform = transforms.Compose([
    transforms.ToTensor(),                 #Convertir une image en tenseur, met les valeurs des pixels entre 0 et 1
    transforms.Normalize((0.5, 0.5, 0.5),  # Moyenne RGB
                         (0.5, 0.5, 0.5))  # Écart-type RGB; pixels deviennent centrés autour de 0
])


trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Définition du CNN de base (3 couches)

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # entrée 3 channels (RGB)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)  # réduit dimension par 2 à chaque fois
        
        # Calculer la taille en sortie avant les couches fully connected
        # CIFAR10 32x32 → après 3 poolings (x2) → 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # après les convolutions on a un tenseur de taille 64*4*4, fc1 les réduit en 128 neurones
        self.fc2 = nn.Linear(128, 10)  # 10 classes CIFAR10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # aplatit en [batch_size, 64*4*4 = 1024]
        x = F.relu(self.fc1(x)) #ou fonction sigmoïde à la fin?
        x = self.fc2(x)
        return x


à entraîner (boucle d'entraînement) training loop + sauvegarde des poids à chaque epoch ; tester pour chaque epoch et voir où l'accuracy est la mieux

Masque pour le dropout

In [5]:
class CNN_MCdropout(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.2, p2=0.2, p3=0.2, p4=0.2):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or [] # si None, aucune couche à masquer
        self.p1 = p1  # dropout sur conv1 output
        self.p2 = p2  # dropout sur conv2 output
        self.p3 = p3  # dropout sur conv3 output
        self.p4 = p4  # dropout sur fc1 output
        # pas de dropout sur la dernière couche
        self.ps = {'conv1': p1, 'conv2': p2, 'conv3': p3, 'fc1': p4}
    
    def dropout_mask(self, x, p, active=True):
        if not self.training or p == 0.0 or not active:
            return torch.ones_like(x) #conserve la dimension de la couche
        mask = (torch.rand_like(x) > p).float() / (1 - p)
        return mask

    def forward(self, x):
        x = F.relu(self.model.conv1(x))
        x = self.dropout_mask(x, self.ps['conv1'], 'conv1' in self.mc_layers) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv2(x))
        x = self.dropout_mask(x, self.ps['conv2'], 'conv2' in self.mc_layers) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv3(x))
        x = self.dropout_mask(x, self.ps['conv3'], 'conv3' in self.mc_layers) * x
        x = self.model.pool(x)

        x = x.view(x.size(0), -1) 
        x = F.relu(self.model.fc1(x))
        x = self.dropout_mask(x, self.ps['fc1'], 'fc1' in self.mc_layers) * x
        x = self.model.fc2(x)
        return x  # pas encore normalisé
    
    # x = x.view ... 
    # x = self.dropout_mask
    # x = F.relu(self.model.fc1(x))
    # x = self.model.fc2(x)

    # faire pareil pour chaque couche concernée, même celles pas concernées, faire le dropout avant relu (pour faire comme pytorch, masque les arêtes entrantes)
    # dans le nn.sequential, prend effet que si model.train()

In [6]:
#Avant ReLU
class CNN_MCdropout_beforeReLU(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.2, p2=0.2, p3=0.2, p4=0.2):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or []
        self.ps = {'conv1': p1, 'conv2': p2, 'conv3': p3, 'fc1': p4}

    def dropout_mask(self, x, p, active=True):
        if not self.training or p == 0.0 or not active:
            return torch.ones_like(x)
        mask = (torch.rand_like(x) > p).float() / (1 - p)
        return mask

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.dropout_mask(x, self.ps['conv1'], 'conv1' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = self.model.conv2(x)
        x = self.dropout_mask(x, self.ps['conv2'], 'conv2' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = self.model.conv3(x)
        x = self.dropout_mask(x, self.ps['conv3'], 'conv3' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = self.model.fc1(x)
        x = self.dropout_mask(x, self.ps['fc1'], 'fc1' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.fc2(x)
        return x


In [7]:
# #Pytorch
# class CNN_MCdropout_torch(nn.Module):
#     def __init__(self, model, mc_layers=None, p1=0.1, p2=0.1, p3=0.1, p4=0.1):
#         super().__init__()
#         self.model = model
#         self.mc_layers = mc_layers or []

#         # On wrappe les couches où on veut du dropout
#         if 'conv1' in self.mc_layers:
#             self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU(), nn.Dropout2d(p1)) #dropout2d pour CNN
#         else:
#             self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU())

#         if 'conv2' in self.mc_layers:
#             self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU(), nn.Dropout2d(p2))
#         else:
#             self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU())

#         if 'conv3' in self.mc_layers:
#             self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU(), nn.Dropout2d(p3))
#         else:
#             self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU())

#         if 'fc1' in self.mc_layers:
#             self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU(), nn.Dropout(p4)) #dropout pour ReLU
#         else:
#             self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU())

#     def forward(self, x):
#         x = self.model.conv1(x)
#         x = self.model.pool(x)

#         x = self.model.conv2(x)
#         x = self.model.pool(x)

#         x = self.model.conv3(x)
#         x = self.model.pool(x)

#         x = x.view(x.size(0), -1)
#         x = self.model.fc1(x)
#         x = self.model.fc2(x)
#         return x


In [8]:
class CNN_MCdropout_torch(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.2, p2=0.2, p3=0.2, p4=0.2):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or []

        # Ajout dropout après conv1
        if 'conv1' in self.mc_layers and isinstance(self.model.conv1, nn.Conv2d):
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU(), nn.Dropout2d(p1))
        else:
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU())

        # Ajout dropout après conv2
        if 'conv2' in self.mc_layers and isinstance(self.model.conv2, nn.Conv2d):
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU(), nn.Dropout2d(p2))
        else:
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU())

        # Ajout dropout après conv3
        if 'conv3' in self.mc_layers and isinstance(self.model.conv3, nn.Conv2d):
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU(), nn.Dropout2d(p3))
        else:
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU())

        # Ajout dropout après fc1
        if 'fc1' in self.mc_layers and isinstance(self.model.fc1, nn.Linear):
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU(), nn.Dropout(p4))
        else:
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU())

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.model.pool(x)

        x = self.model.conv2(x)
        x = self.model.pool(x)

        x = self.model.conv3(x)
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = self.model.fc1(x)
        x = self.model.fc2(x)
        return x

Fonction d'entraînement et test

In [9]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

train_subset, val_subset = random_split(trainset, [train_size, val_size]) # divise le dataset de manière aléatoire en 90/10

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

Fonction d'évaluation (avant entraînement)

In [10]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

Fonction d'entraînement

In [11]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    
    for epoch in range(epochs):#petit nombre d'epochs pour tester (environ 1 minutes pas epoch)
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Évaluer sur validation
        val_loss, val_acc = evaluate(model, valloader, device)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {running_loss/len(trainloader):.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        # Sauvegarde si amélioration
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    
    print("Finished Training")
    return model

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Demande à l'utilisateur quelles couches doivent subir le dropout : ou alors il amène sa liste
user_layers = input(
    "Sur quelles couches voulez-vous appliquer le MC Dropout ? "
    "(choisissez parmi conv1, conv2, conv3, fc1, séparées par des virgules) : ")
mc_layers = [layer.strip() for layer in user_layers.split(',') if layer.strip() in ['conv1','conv2','conv3','fc1']]

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

# On fait une copie pour MC Dropout
base_model_mc = copy.deepcopy(base_model)

model = CNN_MCdropout(base_model_mc, mc_layers=mc_layers, p1=0.2, p2=0.2, p3=0.2, p4=0.2).to(device)
model_beforeReLU = CNN_MCdropout_beforeReLU(base_model_mc, mc_layers=mc_layers, p1=0.2, p2=0.2, p3=0.2, p4=0.2).to(device)
model_torch = CNN_MCdropout_torch(base_model_mc, mc_layers=mc_layers, p1=0.2, p2=0.2, p3=0.2, p4=0.2).to(device)

# Évaluation finale
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")
test_loss2, test_acc2 = evaluate(model_beforeReLU, testloader, device)
print(f"Final Test Loss: {test_loss2:.4f} - Test Acc: {test_acc2:.4f}")
test_loss3, test_acc3 = evaluate(model_torch, testloader, device)
print(f"Final Test Loss: {test_loss3:.4f} - Test Acc: {test_acc3:.4f}")

Chargement du modèle sauvegardé
Final Test Loss: 0.8186 - Test Acc: 0.7267
Final Test Loss: 0.8186 - Test Acc: 0.7267
Final Test Loss: 0.8186 - Test Acc: 0.7267


une fois que j'entraîne le modèle je sauvegarde les poids au format pickle, puis model_orig.loadstatedict (?) puis mettre le chemin de mes poids

MC Dropout prédiction

In [13]:
def mc_predict_mean_probs(model, X, T=1000, verbose=True):
    model.train()  # activer dropout pendant l'inférence MC
    probs_list = []
    start_time = time.time() 
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            logits_t = model(X)
            probs_t = F.softmax(logits_t, dim=1)
            probs_list.append(probs_t.unsqueeze(0))
    elapsed = time.time() - start_time
    probs_mc = torch.cat(probs_list, dim=0)       
    model.eval()  # remettre le modèle en mode eval à la fin
    if verbose:
        print(f"Temps total: {elapsed:.2f} s  |  Temps moyen par passe: {elapsed/T:.4f} s")
    return probs_mc.mean(0), elapsed                        

je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [14]:
# Test MC Dropout sur un batch
X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)
probs, t1 = mc_predict_mean_probs(model, X, T=1000, verbose=True)
probs2, t2 = mc_predict_mean_probs(model_beforeReLU, X, T=1000, verbose=True)
probs3, t3 = mc_predict_mean_probs(model_torch, X, T=1000, verbose=True)
Y_hat = probs.argmax(1)
Y_hat2 = probs2.argmax(1)
Y_hat3 = probs3.argmax(1)

print("Classes vraies       :", Y.tolist())
print(f"After ReLU   (t={t1:.2f}s): {Y_hat.tolist()}")
print(f"Before ReLU  (t={t2:.2f}s): {Y_hat2.tolist()}")
print(f"Torch Dropout (t={t3:.2f}s): {Y_hat3.tolist()}")


100%|██████████| 1000/1000 [00:39<00:00, 25.13it/s]


Temps total: 39.80 s  |  Temps moyen par passe: 0.0398 s


100%|██████████| 1000/1000 [00:40<00:00, 24.56it/s]


Temps total: 40.72 s  |  Temps moyen par passe: 0.0407 s


100%|██████████| 1000/1000 [00:13<00:00, 76.36it/s]

Temps total: 13.11 s  |  Temps moyen par passe: 0.0131 s
Classes vraies       : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3]
After ReLU   (t=39.80s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 1, 4, 2, 4, 0, 9, 6, 3, 5, 2, 3, 9, 3, 4, 1, 9, 5, 4, 6, 3, 6, 0, 9, 5, 3, 7, 6, 9, 8, 6, 3, 8, 8, 5, 5, 5, 3, 7, 5, 6, 3, 6, 2, 1, 0, 3, 7, 0, 6, 8, 8, 8, 2, 2, 3, 3, 8, 8, 1, 1, 7, 2, 2, 2, 8, 8, 9, 0, 2, 8, 6, 4, 6, 6, 0, 0, 3, 2, 5, 6, 3, 1, 1, 2, 6, 8, 5, 6, 0, 2, 2, 9, 3, 0, 4, 3, 5, 8, 2, 1, 2, 8, 0, 8, 3]
Before ReLU  (t=40.72s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 1, 4, 2, 4, 0, 9, 6, 6, 5, 2, 

rajouter en argument de generate_mc_outputs measure = variance (en gros on laisse à l'utilisateur le choix de la mesure d'incertitude)

In [15]:
# def generate_mc_outputs(model, X, T=1000, metrics="mc_estimate", labels=None, verbose=True):
#     model.train()  # dropout actif en inférence
#     outputs = []
#     start_forward = time.time()
#     with torch.no_grad():
#         for _ in tqdm(range(T), disable=not verbose):
#             out = model(X)                      
#             outputs.append(out.unsqueeze(0))    
#     elapsed_forward = time.time() - start_forward
    
#     outputs = torch.cat(outputs, dim=0)         
#     all_probs   = torch.softmax(outputs, dim=2)  
#     mean_probs  = all_probs.mean(dim=0)          # moyenne des softmax
#     var_pred  = outputs.var(dim=0)                # variance des logits
   
#     results = {}
#     elapsed_metrics = {}
#     for metric in metrics:
#         start_metric = time.time()
#         if metric == "mc_estimate":
#             results[metric] = mean_probs
#         elif metric == "variance":
#             results[metric] = var_pred
#         elif metric == "predictive_entropy":
#             results[metric] = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1) # +1e-12 pour éviter log(0)
#         elif metric == "relative_norm":
#             if labels is None:
#                 raise ValueError("labels must be provided for relative_norm metric")
#             labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
#             diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)   # ||ŷ̄ - Y||
#             denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
#             results[metric] = diff_norm / (denom + 1e-12)                  
#         else:
#             raise ValueError(f"Metric {metric} non reconnue")
#         elapsed_metrics[metric] = time.time() - start_metric
#     model.eval()  # remettre le modèle en mode eval à la fin

#     if verbose:
#         print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
#         for m, t in elapsed_metrics.items():
#             print(f"Temps calcul métrique '{m}': {t:.6f}s")

#     return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


In [16]:
def generate_mc_outputs(model, X, T=1000, metrics=["mc_estimate"], labels=None, verbose=True):
    model.train()
    outputs = []

    # --- forward pass ---
    start_forward = time.time()
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            out = model(X)
            outputs.append(out.unsqueeze(0))
    elapsed_forward = time.time() - start_forward

    outputs = torch.cat(outputs, dim=0)
    results = {}
    elapsed_metrics = {}

    # --- calcul des métriques avec chrono réel ---
    for metric in metrics:
        start_metric = time.time()
        if metric == "mc_estimate":
            all_probs = torch.softmax(outputs, dim=2)
            results[metric] = all_probs.mean(dim=0)
        elif metric == "variance":
            results[metric] = outputs.var(dim=0)
        elif metric == "predictive_entropy":
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            results[metric] = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)
        elif metric == "relative_norm":
            if labels is None:
                raise ValueError("labels doivent être fournis pour relative_norm")
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
            diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)
            denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
            results[metric] = diff_norm / (denom + 1e-12)
        else:
            raise ValueError(f"Métrique {metric} non reconnue")
        elapsed_metrics[metric] = time.time() - start_metric

    model.eval()

    if verbose:
        print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
        for m, t in elapsed_metrics.items():
            print(f"Temps calcul métrique '{m}': {t:.6f}s")

    return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


le mc estimate n'est pas une mesure d'incertitude, sert pour construire une autre variable

Métriques avec la méthode after ReLU

In [17]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 39.13 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.013109 s
Résultat : tensor([[0.0887, 0.0489, 0.0663,  ..., 0.0162, 0.1834, 0.0381],
        [0.0753, 0.3606, 0.0173,  ..., 0.0080, 0.4527, 0.0233],
        [0.1724, 0.1474, 0.0704,  ..., 0.0386, 0.3182, 0.0764],
        ...,
        [0.3297, 0.1223, 0.1991,  ..., 0.0094, 0.1169, 0.0856],
        [0.2316, 0.0279, 0.1366,  ..., 0.0180, 0.2893, 0.0253],
        [0.0411, 0.0162, 0.0771,  ..., 0.0541, 0.0317, 0.0478]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.008040 s
Résultat : tensor([[ 48.0755,  55.5773,  31.1870,  ...,  45.8058,  63.9684,  80.2050],
        [ 45.8953,  90.0559,  62.0633,  ..., 107.5315,  77.4674,  56.4919],
        [ 24.2500,  48.6968,  27.0492,  ...,  47.2450,  37.1087,  36.7127],
        ...,
        [ 54.8115,  70.9063,  50.9

Métriques avec la méthode before ReLU

In [18]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model_beforeReLU, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 38.08 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.009809 s
Résultat : tensor([[0.0638, 0.0481, 0.0814,  ..., 0.0127, 0.1639, 0.0415],
        [0.0694, 0.3589, 0.0233,  ..., 0.0030, 0.4506, 0.0338],
        [0.1653, 0.1542, 0.0695,  ..., 0.0372, 0.3206, 0.0764],
        ...,
        [0.2894, 0.1558, 0.1988,  ..., 0.0119, 0.1053, 0.0810],
        [0.2399, 0.0251, 0.1286,  ..., 0.0169, 0.3009, 0.0191],
        [0.0398, 0.0207, 0.0657,  ..., 0.0501, 0.0317, 0.0480]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.006049 s
Résultat : tensor([[43.5335, 59.9560, 33.1830,  ..., 41.6648, 64.7728, 83.7081],
        [44.0530, 81.6642, 63.4789,  ..., 99.5182, 76.3568, 55.0732],
        [24.5917, 51.9879, 31.2037,  ..., 45.7308, 37.6929, 35.1566],
        ...,
        [52.5046, 71.2266, 53.9730,  ..., 96.4865, 6

Métriques avec la méthode PyTorch

In [19]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model_torch, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 12.13 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.000000 s
Résultat : tensor([[0.0666, 0.0375, 0.0616,  ..., 0.0074, 0.1438, 0.0295],
        [0.0635, 0.3449, 0.0131,  ..., 0.0049, 0.4886, 0.0186],
        [0.1781, 0.0924, 0.0524,  ..., 0.0288, 0.3759, 0.0588],
        ...,
        [0.3630, 0.1076, 0.2202,  ..., 0.0029, 0.1148, 0.0601],
        [0.2571, 0.0136, 0.1228,  ..., 0.0108, 0.2847, 0.0166],
        [0.0264, 0.0136, 0.0505,  ..., 0.0516, 0.0209, 0.0218]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.000000 s
Résultat : tensor([[11.6176, 16.7505,  9.0121,  ..., 13.2088, 19.1698, 22.5741],
        [11.9144, 26.7601, 15.0670,  ..., 28.3169, 22.5377, 15.5273],
        [ 6.8916, 13.3076,  7.0973,  ..., 11.3614, 11.6382, 10.9585],
        ...,
        [13.0542, 21.8755, 14.0816,  ..., 25.4506, 1

pour la fonction de confiance, dois-je faire 1/métrique choisie? ou ça marche qu'avec la variance?

oui si c'est dans R+ : 1/ ? exp(-...) si c'est pour avoir entre 0 et 1 ; quitte à normaliser après